# SemEval 2026 MWAHAHA - Colab Generation

Use this notebook to run generation on Google Colab with GPU.

Recommended runtime: `Runtime > Change runtime type > T4 GPU`.

## 1. Clone The GitHub Repository

The notebook runs from a fresh clone of the GitHub repository inside the Colab runtime.

In [ ]:
REPO_URL = 'https://github.com/AgnesePino/LLM-Project-SemEval-Humor-Generation.git'
PROJECT_DIR = '/content/LLM-Project-SemEval-Humor-Generation'

!test -d "$PROJECT_DIR" || git clone "$REPO_URL" "$PROJECT_DIR"
%cd $PROJECT_DIR
!git pull

## 2. Install Dependencies

This installs the local package plus GPU generation dependencies.

In [ ]:
!pip install -q -e ".[colab]"

## 3. Optional HuggingFace Login

Llama models usually require accepting the model license on HuggingFace and logging in.

In [ ]:
# Uncomment if needed, then paste your HuggingFace token.
# from huggingface_hub import login
# login()

## 4. Convert Input Data

Use an official-style TSV with columns: `id`, `word1`, `word2`, `headline`.

For Word Inclusion rows, both `word1` and `word2` must be present.

In [ ]:
INPUT_TSV = 'data/samples/sample_task_a.tsv'
CONVERTED_JSONL = 'data/prepared/colab_input.jsonl'

!PYTHONPATH=src python3 -m data_preparation.prepare_task_a \
  --input "$INPUT_TSV" \
  --output "$CONVERTED_JSONL"

## 5. Run Generation

Available presets: `llama`, `qwen`, `mistral`.

Start with `qwen` if Llama access is not configured yet.

In [ ]:
MODEL = 'qwen'  # llama, qwen, or mistral
OUTPUT_JSONL = f'data/generated/base/{MODEL}_colab_predictions.jsonl'

!PYTHONPATH=src python3 -m generation.base.generate_hf \
  --model "$MODEL" \
  --input "$CONVERTED_JSONL" \
  --output "$OUTPUT_JSONL"

## 6. Preview Outputs

In [ ]:
!head -5 "$OUTPUT_JSONL"

## 7. Validate Constraint Satisfaction

In [ ]:
REPORT_JSON = f'data/evaluated/base/{MODEL}_validation_report.json'

!PYTHONPATH=src python3 -m evaluation.validate_outputs \
  --input "$CONVERTED_JSONL" \
  --predictions "$OUTPUT_JSONL" \
  --report "$REPORT_JSON"

!cat "$REPORT_JSON"